# Integrated Stage 3 model evaluation

This notebook refits auto-ARIMA, VAR, GJR-GARCH, asymmetric EGARCH, Student-t GARCH, VIX-GARCH-X, and quantile GARCH at each Stage 3 origin, and evaluates them together with the existing cached baseline and predictor-model forecasts on common origins. This full evaluation can take substantially longer than the old cached-only run.

In [1]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

cwd = Path.cwd()
repo_root = next(
    (candidate for candidate in (cwd, *cwd.parents)
     if (candidate / 'pyproject.toml').is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError('Could not locate repository root.')
sys.path.insert(0, str(repo_root / 'src'))

from gold_vol_pipeline.evaluation import qlike, summary_table
from gold_vol_pipeline.regression_models import PREDICTOR_COLUMNS

data_path = repo_root / 'data/processed/stage2_multivariate.csv'
manifest_path = repo_root / 'data/processed/stage2_multivariate_manifest.json'
if not data_path.is_file():
    raise FileNotFoundError('Run the Stage 2 predictor pipeline before this notebook.')
data = pd.read_csv(data_path, parse_dates=['date', 'target_end_date'])
manifest = json.loads(manifest_path.read_text(encoding='utf-8')) if manifest_path.is_file() else {}
horizon = int(manifest.get('horizon', 5))
if manifest.get('target', 'realized_variance') != 'realized_variance':
    raise ValueError('This evaluation requires the realized_variance target.')
required = {'date', 'log_return', 'realized_variance', 'target_end_date', *PREDICTOR_COLUMNS}
missing = sorted(required - set(data.columns))
if missing:
    raise ValueError(f'Stage 2 data is missing required columns: {missing}')
data = data.sort_values('date').reset_index(drop=True)
print(f'Input rows: {len(data):,}; dates: {data.date.min().date()} through {data.date.max().date()}')
print(f'Target: {horizon}-session forward realized variance.')

Input rows: 2,976; dates: 2015-01-05 through 2026-11-03
Target: 5-session forward realized variance.


## Integrated origin-wise evaluation

The seven specified candidates are refitted together; previously completed baselines and HAR/HARX, lasso and ridge forecasts are reused. The HK order is selected using only the first origin and held fixed. GARCH-X holds the last observed VIX constant during the forecast horizon. Failures are retained in the attempts file and the comparison uses common successful origins.

In [2]:
from gold_vol_pipeline.stage3_evaluation import evaluate_stage3

if horizon != 5:
    raise ValueError('This Stage 3 evaluation requires the five-session target.')
results, summary, evaluation_manifest = evaluate_stage3(
    data_path,
    repo_root / 'data/processed/evaluation_forecasts_with_stage1.csv',
    repo_root / 'data/processed',
    workers=4,
)
print(json.dumps(evaluation_manifest, indent=2))
display(summary)

/Users/enoch/Documents/Gold-vol-forecasting_ECON4120_GroupProject/src/gold_vol_pipeline/auto_arima.py:104: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  _, p_value, *_ = kpss(differenced, regression="c", nlags="auto")
/Users/enoch/Documents/Gold-vol-forecasting_ECON4120_GroupProject/src/gold_vol_pipeline/auto_arima.py:104: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  _, p_value, *_ = kpss(differenced, regression="c", nlags="auto")


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.wa

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/optimize/_slsqp_py.py:435: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  fx = wrapped_fun(x)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/optimize/_slsqp_py.py:439: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  g = append(wrapped_grad(x), 0.0)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/optimize/_slsqp_py.py:499: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  a_ieq = vstack([con['jac'](x, *con['args'])
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/optimize/_slsqp_py.py:435: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  fx = wrapped_fun(x)
/Library/Frameworks/Python.framework/Versions/3.13/lib/pyth

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/optimize/_slsqp_py.py:435: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  fx = wrapped_fun(x)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/optimize/_slsqp_py.py:439: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  g = append(wrapped_grad(x), 0.0)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/optimize/_slsqp_py.py:499: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  a_ieq = vstack([con['jac'](x, *con['args'])
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/optimize/_slsqp_py.py:435: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  fx = wrapped_fun(x)


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/optimize/_slsqp_py.py:435: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  fx = wrapped_fun(x)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/optimize/_slsqp_py.py:439: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  g = append(wrapped_grad(x), 0.0)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/optimize/_slsqp_py.py:499: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  a_ieq = vstack([con['jac'](x, *con['args'])
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/optimize/_slsqp_py.py:435: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  fx = wrapped_fun(x)
/Library/Frameworks/Python.framework/Versions/3.13/lib/pyth

... (386 outputs left out)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/optimize/_slsqp_py.py:435: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  fx = wrapped_fun(x)


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/optimize/_slsqp_py.py:435: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  fx = wrapped_fun(x)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/optimize/_slsqp_py.py:439: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  g = append(wrapped_grad(x), 0.0)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/optimize/_slsqp_py.py:499: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  a_ieq = vstack([con['jac'](x, *con['args'])


{
  "panel_sha256": "a2f957950e4df6d445271b720ac758a4eb777d998e4e37e0f1195229c515afcd",
  "cached_forecasts_sha256": "d3cc22bc2dfb29c6bfc0e2274738b221172cf2c116e5feea5ec2ef3c2a570f4a",
  "origin_count": 445,
  "common_origin_count": 443,
  "origin_start": "2018-01-05",
  "origin_end": "2026-11-03",
  "refitted_models": [
    "auto_arima_hk",
    "var",
    "gjr_garch",
    "egarch_asym",
    "garch_t",
    "garchx_vix",
    "quantile_garch"
  ],
  "cached_models": [
    "ar1_squared_return",
    "arch1",
    "ewma",
    "ewma_fixed",
    "garch11",
    "har",
    "harx_dxy_vix",
    "harx_full",
    "harx_vix",
    "historical_variance",
    "lasso_dxy_vix",
    "lasso_full",
    "lasso_vix",
    "ma1_squared_return",
    "ridge_dxy_vix",
    "ridge_full",
    "ridge_vix",
    "rolling_variance"
  ],
  "fixed_arima_order": [
    3,
    1,
    1
  ],
  "arima_selection_origin": "2018-01-05",
  "failure_counts": {
    "auto_arima_hk": 0,
    "egarch_asym": 0,
    "garch_t": 0,
    "garch

,model,n,MSE,MAE,QLIKE,MSE_ratio_to_benchmark,QLIKE_ratio_to_benchmark,DM_MSE_p_raw,DM_MSE_p_holm,DM_QLIKE_p_raw,DM_QLIKE_p_holm,MZ_a,MZ_b,MZ_wald_p,RMSE
0,garchx_vix,443,9.492038e-07,0.000420,0.455247,0.838513,0.715305,0.030720,0.905299,2.072528e-02,8.659454e-01,-0.000024,1.228912,1.481925e-01,0.000974
1,egarch_asym,443,9.602097e-07,0.000413,0.443171,0.848235,0.696330,0.049734,1.000000,1.369310e-02,6.298827e-01,0.000017,1.148472,1.098894e-01,0.000980
2,gjr_garch,443,9.636184e-07,0.000418,0.454524,0.851246,0.714169,0.068492,1.000000,2.189254e-02,8.659454e-01,0.000112,0.967378,2.886967e-02,0.000982
3,garch_t,443,9.735968e-07,0.000431,0.456494,0.860061,0.717264,0.079170,1.000000,2.241024e-02,8.659454e-01,0.000160,0.796445,1.349272e-02,0.000987
4,garch11,443,9.739768e-07,0.000423,0.455396,0.860397,0.715538,0.064359,1.000000,1.867334e-02,8.403003e-01,0.000172,0.811071,6.952814e-03,0.000987
5,quantile_garch,443,9.739995e-07,0.000423,0.455578,0.860417,0.715825,0.064462,1.000000,1.880632e-02,8.403003e-01,0.000172,0.810398,6.634875e-03,0.000987
6,ewma,443,9.803330e-07,0.000444,0.452554,0.866012,0.711073,0.124141,1.000000,2.554364e-02,8.659454e-01,0.000174,0.742176,6.218921e-03,0.000990
7,har,443,9.857301e-07,0.000418,0.466283,0.870779,0.732645,0.045566,1.000000,2.205394e-02,8.659454e-01,0.000019,1.159905,9.604148e-02,0.000993
8,lasso_vix,443,9.891328e-07,0.000430,0.457634,0.873785,0.719055,0.150663,1.000000,2.013827e-02,8.659454e-01,0.000249,0.691286,2.979555e-03,0.000995
9,ewma_fixed,443,9.913842e-07,0.000454,0.456870,0.875774,0.717855,0.162067,1.000000,4.714960e-02,1.000000e+00,0.000224,0.652388,2.905166e-06,0.000996


In [3]:
dm_mse = summary.attrs['dm_pvalue_matrix_mse']
dm_qlike = summary.attrs['dm_pvalue_matrix_qlike']
display(dm_qlike.style.format('{:.3g}'))

,garchx_vix,egarch_asym,gjr_garch,garch_t,garch11
garchx_vix,1,0.746,0.985,0.97,0.996
egarch_asym,0.746,1,0.0831,0.324,0.353
gjr_garch,0.985,0.0831,1,0.868,0.932
garch_t,0.97,0.324,0.868,1,0.863
garch11,0.996,0.353,0.932,0.863,1


## Full versus reduced predictor sets

Compare full seven-variable models against a VIX-only set (the Yahoo/FRED screen's significant predictor) and a DXY-plus-VIX set (both individually significant in the separate LSEG screen). DXY-plus-VIX is a cross-source sensitivity set, not a significance claim for the Yahoo/FRED sample. Each model family uses identical forecast origins and its full-set version as the Diebold-Mariano benchmark; report loss ratios and Holm-adjusted MSE/QLIKE p-values within each family. These comparisons evaluate predictive performance rather than selecting variables by coefficient p-values alone.

In [4]:
predictor_comparisons = {
    'HARX': ['harx_full', 'harx_dxy_vix', 'harx_vix'],
    'Lasso': ['lasso_full', 'lasso_dxy_vix', 'lasso_vix'],
    'Ridge': ['ridge_full', 'ridge_dxy_vix', 'ridge_vix'],
}
comparison_tables = []
for family, model_names in predictor_comparisons.items():
    family_results = results.loc[results['model'].isin(model_names)].copy()
    family_summary = summary_table(
        family_results,
        benchmark=model_names[0],
        horizon=horizon,
        top_n=len(model_names),
    )
    family_summary.insert(0, 'family', family)
    family_summary.attrs = {}
    comparison_tables.append(family_summary)
predictor_comparison = pd.concat(comparison_tables, ignore_index=True)
comparison_columns = [
    'family', 'model', 'n', 'MSE', 'MAE', 'QLIKE',
    'MSE_ratio_to_benchmark', 'QLIKE_ratio_to_benchmark',
    'DM_MSE_p_holm', 'DM_QLIKE_p_holm',
]
display(predictor_comparison[comparison_columns].style.format({
    'MSE': '{:.6g}',
    'MAE': '{:.6g}',
    'QLIKE': '{:.4f}',
    'MSE_ratio_to_benchmark': '{:.3f}',
    'QLIKE_ratio_to_benchmark': '{:.3f}',
    'DM_MSE_p_holm': '{:.4g}',
    'DM_QLIKE_p_holm': '{:.4g}',
}).hide(axis='index'))
comparison_path = repo_root / 'data/processed/predictor_set_comparison.csv'
predictor_comparison[comparison_columns].to_csv(comparison_path, index=False)
print(f'Wrote predictor-set comparison to {comparison_path}')

family,model,n,MSE,MAE,QLIKE,MSE_ratio_to_benchmark,QLIKE_ratio_to_benchmark,DM_MSE_p_holm,DM_QLIKE_p_holm
HARX,harx_vix,443,9.94439e-07,0.000432249,0.4575,0.776,1.016,1,1
HARX,harx_dxy_vix,443,1.01995e-06,0.000437778,0.4601,0.795,1.022,1,1
HARX,harx_full,443,1.28225e-06,0.000459082,0.4504,1.000,1.000,1,1
Lasso,lasso_vix,443,9.89133e-07,0.000430021,0.4576,0.978,0.966,0.8955,0.7404
Lasso,lasso_full,443,1.01187e-06,0.00042309,0.4740,1.000,1.000,1,1
Lasso,lasso_dxy_vix,443,1.021e-06,0.000434457,0.4642,1.009,0.979,0.8955,0.8955
Ridge,ridge_vix,443,9.94207e-07,0.000431209,0.4580,0.887,1.017,0.8616,1
Ridge,ridge_dxy_vix,443,1.03943e-06,0.00043791,0.4633,0.927,1.029,0.9227,1
Ridge,ridge_full,443,1.12099e-06,0.000438405,0.4503,1.000,1.000,1,1


Wrote predictor-set comparison to /Users/enoch/Documents/Gold-vol-forecasting_ECON4120_GroupProject/data/processed/predictor_set_comparison.csv


In [5]:
plot_dir = repo_root / 'plots/final_evaluation'
plot_dir.mkdir(parents=True, exist_ok=True)
valid_summary = summary.loc[summary['n'] > 0]
best_models = valid_summary['model'].head(3).tolist()
benchmark = 'historical_variance'
plot_models = list(dict.fromkeys([*best_models, benchmark]))

# Actual versus forecast for the three lowest-MSE models and the historical-variance benchmark.
wide = results[results.model.isin(plot_models)].pivot(index='origin_date', columns='model', values='forecast')
actual = results[results.model == benchmark].set_index('origin_date')['actual']
figure, axis = plt.subplots(figsize=(13, 6))
axis.plot(actual.index, actual, label='Observed', color='black', linewidth=1.5)
for name in plot_models:
    if name in wide:
        axis.plot(wide.index, wide[name], label=name, alpha=0.85)
axis.set_title('5-session forward realized variance, observed vs forecast')
axis.set_xlabel('Forecast origin date')
axis.set_ylabel('Realized variance (squared log-return units; 5 sessions)')
axis.legend(ncol=2)
figure.tight_layout()
figure.savefig(plot_dir / 'actual_vs_forecast.png', dpi=200)
plt.close(figure)

# Cumulative QLIKE loss difference: negative values favor the candidate.
benchmark_rows = results[results.model == benchmark][['origin_date', 'actual', 'forecast']].rename(columns={'forecast': 'benchmark'})
figure, axis = plt.subplots(figsize=(13, 6))
for name in plot_models:
    if name == benchmark:
        continue
    model_rows = results[results.model == name][['origin_date', 'actual', 'forecast']]
    aligned = model_rows.merge(benchmark_rows, on=['origin_date', 'actual'], validate='one_to_one').dropna()
    if len(aligned) and (aligned[['actual', 'forecast', 'benchmark']] > 0).all().all():
        model_loss, _ = qlike(aligned.actual, aligned.forecast)
        benchmark_loss, _ = qlike(aligned.actual, aligned.benchmark)
        axis.plot(aligned.origin_date, (model_loss - benchmark_loss).cumsum(), label=name)
axis.axhline(0, color='black', linewidth=0.8)
axis.set_title(f'Cumulative QLIKE difference vs {benchmark}')
axis.set_xlabel('Forecast origin date')
axis.set_ylabel('Cumulative QLIKE loss difference (candidate − benchmark; 5 sessions)')
axis.legend()
figure.tight_layout()
figure.savefig(plot_dir / 'cumulative_qlike_vs_historical_variance.png', dpi=200)
plt.close(figure)

# Selected lasso coefficients across origins, with selection frequency.
lasso_rows = results[(results.model == 'lasso_full') & results.coefficients.notna()].copy()
coefficient_rows = []
for row in lasso_rows.itertuples():
    for feature, coefficient in json.loads(row.coefficients).items():
        coefficient_rows.append({'origin_date': row.origin_date, 'feature': feature, 'coefficient': coefficient})
coefficient_frame = pd.DataFrame(coefficient_rows)
figure, axes = plt.subplots(2, 1, figsize=(13, 9), gridspec_kw={'height_ratios': [2, 1]})
if not coefficient_frame.empty:
    frequencies = coefficient_frame.groupby('feature').origin_date.nunique().div(lasso_rows.origin_date.nunique()).sort_values(ascending=False)
    selected_features = frequencies.head(8).index.tolist()
    for feature in selected_features:
        path = coefficient_frame[coefficient_frame.feature == feature]
        axes[0].plot(path.origin_date, path.coefficient, label=feature)
    axes[0].legend(ncol=2)
    axes[1].bar(frequencies.head(12).index, frequencies.head(12).values)
    axes[1].tick_params(axis='x', rotation=45)
axes[0].set_title('Lasso selected coefficient paths across origins')
axes[0].set_ylabel('Log-variance coefficient (training-standardized predictors; 5 sessions)')
axes[0].set_xlabel('Forecast origin date')
axes[1].set_title('Lasso coefficient selection frequency (5-session target)')
axes[1].set_ylabel('Fraction of fitted origins')
axes[1].set_xlabel('Feature')
figure.tight_layout()
figure.savefig(plot_dir / 'lasso_coefficients.png', dpi=200)
plt.close(figure)

# Pairwise DM QLIKE p-values for the top models.
figure, axis = plt.subplots(figsize=(8, 7))
image = axis.imshow(dm_qlike.to_numpy(dtype=float), vmin=0, vmax=1, cmap='viridis_r')
axis.set_xticks(range(len(dm_qlike.columns)), dm_qlike.columns, rotation=45, ha='right')
axis.set_yticks(range(len(dm_qlike.index)), dm_qlike.index)
axis.set_title('Pairwise Diebold-Mariano p-values (QLIKE; 5-session horizon)')
figure.colorbar(image, ax=axis, label='Two-sided p-value')
figure.tight_layout()
figure.savefig(plot_dir / 'dm_qlike_heatmap.png', dpi=200)
plt.close(figure)
print(f'Saved 200 dpi figures to {plot_dir}')

Saved 200 dpi figures to /Users/enoch/Documents/Gold-vol-forecasting_ECON4120_GroupProject/plots/final_evaluation


## Model-comparison handoff

The integrated comparison combines the refitted candidates with cached forecasts on their shared successful origins. Failed fits are recorded separately. The older 500-origin Stage 1 results use a different sample and must not be ranked directly against this table.

Interpret loss, Diebold-Mariano, and Mincer-Zarnowitz evidence alongside overlapping-target dependence, model-selection uncertainty, and source-timing limitations.